# Bài 4. Alpha-Beta với đánh giá heuristic

`cutoff` đếm số nước từ gốc. Tại lá kết thúc, dùng utility chính xác. Ở nút chưa kết thúc tại độ sâu cắt, `eval_fun(state, player)` tính 0,4 nhân với hiệu giữa số hàng, cột hoặc đường chéo có hai quân và một ô trống của mình và của đối thủ. Điểm được chặn trong [−0,8, 0,8] để điểm thắng và thua chắc chắn giữ giá trị biên. Dùng thứ tự giữa–góc–cạnh cho cả ba cấu hình.

In [1]:
from time import perf_counter
from tictactoe import other, result
from tictactoe_search import alpha_beta_search
from lab05_experiments import CASES, benchmark, game_value

def measure(search, board, player, **settings):
    start = perf_counter()
    record = search(board, player, **settings)
    return dict(record, ms=(perf_counter() - start) * 1000)

def exact_move_value(board, player, move):
    child = result(board, player, move)
    return -game_value(tuple(child), other(player))

def print_match(summary):
    for label, row in [*summary["by_role"].items(), ("Tổng", summary)]:
        print(
            f'{label:>4} {row["games"]:>4} {row["wins"]:>6} '
            f'{row["draws"]:>5} {row["losses"]:>5} {row["ms_per_move"]:>13.3f}'
        )

In [2]:
from tictactoe import empty_board, play
from tictactoe_search import eval_fun, alpha_beta_player, heuristic2_player, heuristic4_player

open_two = list("xx  o    ")
terminal_x = list("xxxoo    ")
for name, board in [("Bàn trống", empty_board()), ("X có một đường mở", open_two), ("X thắng", terminal_x)]:
    print(f"{name:<21} X: {eval_fun(board, 'x')}  O: {eval_fun(board, 'o')}")

assert eval_fun(open_two, "x") == (0.4, False)
assert eval_fun(terminal_x, "o") == (-1, True)

Bàn trống             X: (0.0, False)  O: (0.0, False)
X có một đường mở     X: (0.4, False)  O: (-0.4, False)
X thắng               X: (1, True)  O: (-1, True)


Hai góc nhìn cho điểm đối dấu. Một đường mở của X cho +0,4 từ góc nhìn X, còn một hàng X đã hoàn thành cho +1 và cờ kết thúc bằng `True`.

## So sánh cùng các trạng thái của Bài 3

Cột «Điểm» là giá trị thuật toán trả ở gốc. Cột «Điểm thật» là giá trị của nước đã chọn khi hai bên chơi tối ưu sau đó. Cột cuối so điểm thật với giá trị của nước tối ưu để đánh giá quyết định. Hai nước khác nhau có thể cùng điểm thật. Thời gian đo và cách đếm nút giống Bài 3.

In [3]:
configurations = (("Đầy đủ", None), ("cutoff 2", 2), ("cutoff 4", 4))

def compare(board, player):
    records = [
        measure(alpha_beta_search, board, player, ordered=True, cutoff=cutoff)
        for _, cutoff in configurations
    ]
    exact = records[0]["value"]
    for row in records:
        row["exact"] = exact_move_value(board, player, row["move"])
        row["optimal"] = row["exact"] == exact
    return records

def show_comparison(label, records):
    for (name, _), row in zip(configurations, records):
        note = "Tối ưu" if row["optimal"] else "Dưới tối ưu"
        print(
            f'{label:<28} {name:<10} {row["move"]:>3} {row["value"]:>6.1f} '
            f'{row["nodes"]:>8,} {row["ms"]:>10.3f} {row["exact"]:>10} {note:>13}'
        )

case_rows = [compare(board, player) for _, board, player in CASES]
print(
    f'{"Trạng thái":<28} {"Cấu hình":<10} {"Ô":>3} {"Điểm":>6} '
    f'{"Nút":>8} {"ms":>10} {"Điểm thật":>10} {"Nhận xét":>13}'
)
for (label, _, _), rows in zip(CASES, case_rows):
    show_comparison(label, rows)

Trạng thái                   Cấu hình     Ô   Điểm      Nút         ms  Điểm thật      Nhận xét
X thắng ngay                 Đầy đủ       2    1.0       44      5.375          1        Tối ưu
X thắng ngay                 cutoff 2     8    1.0       13      1.777          1        Tối ưu
X thắng ngay                 cutoff 4     2    1.0       36      6.326          1        Tối ưu
X phải chặn                  Đầy đủ       7    0.0      149     22.416          0        Tối ưu
X phải chặn                  cutoff 2     7   -0.4       26      3.108          0        Tối ưu
X phải chặn                  cutoff 4     7    0.0      120     16.400          0        Tối ưu
X bất lợi (giả định)         Đầy đủ       4   -1.0       51      6.422         -1        Tối ưu
X bất lợi (giả định)         cutoff 2     4   -0.8       17      1.794         -1        Tối ưu
X bất lợi (giả định)         cutoff 4     4   -1.0       35      3.840         -1        Tối ưu
Bàn trống                    Đầy đủ     

Ở bàn trống, cắt tại 2 và 4 nước duyệt lần lượt 26 và 303 nút, so với 7,275 nút của Alpha-Beta đầy đủ. Hai độ sâu cắt chọn nước tối ưu ở lần lượt 5/5 và 5/5 trạng thái đã kiểm tra.

### Thế bổ sung để kiểm tra đòn hai đường thắng

X ở ô 0 và 5, O ở ô 1 và 3. X sắp đi.

In [4]:
from lab05_experiments import make_board, move_values

fork_board = make_board(x=(0, 5), o=(1, 3))
fork_records = compare(fork_board, "x")
show_comparison("X có thể tạo hai đường thắng", fork_records)
print("Điểm chính xác theo ô:", move_values(fork_board, "x"))
assert fork_records[1]["exact"] == 0
assert fork_records[2]["exact"] == 1

X có thể tạo hai đường thắng Đầy đủ       8    1.0      101      8.250          1        Tối ưu
X có thể tạo hai đường thắng cutoff 2     4    0.0       14      1.264          0   Dưới tối ưu
X có thể tạo hai đường thắng cutoff 4     8    1.0       75      6.190          1        Tối ưu
Điểm chính xác theo ô: {2: 0, 4: 0, 6: 0, 7: 0, 8: 1}


Cutoff 2 chọn ô 4 với giá trị thật 0. Cutoff 4 chọn ô 8, sau đó X có hai cách thắng ở ô 2 hoặc ô 4. O chỉ kịp chặn một ô. Độ sâu 4 nhìn thấy lượt thắng của X sau nước chặn của O nên tìm được nước có giá trị thật +1.

## Đấu với Random và kiểm tra chéo

Mỗi độ sâu chơi 100 ván ở vai X và 100 ván ở vai O. Sau đó, ba tác tử xác định đấu từng cặp ở cả hai cách phân vai. Mỗi cách phân vai xác định chỉ cần một ván từ bàn trống vì chuỗi nước lặp lại nếu chơi thêm.

In [5]:
heuristic_metrics = {}
for name, agent in [("cutoff 2", heuristic2_player), ("cutoff 4", heuristic4_player)]:
    summary = benchmark(agent)
    heuristic_metrics[name] = summary
    print(name)
    print(f'{"Vai":>4} {"Ván":>4} {"Thắng":>6} {"Hòa":>5} {"Thua":>5} {"ms/nước":>13}')
    print_match(summary)

head_to_head = []
for x, o in [
    (heuristic2_player, alpha_beta_player), (alpha_beta_player, heuristic2_player),
    (heuristic4_player, alpha_beta_player), (alpha_beta_player, heuristic4_player),
    (heuristic2_player, heuristic4_player), (heuristic4_player, heuristic2_player),
]:
    outcome = play(x, o, N=1)
    winner = next(mark for mark, count in outcome.items() if count == 1)
    head_to_head.append(winner)
    print(f"X={x.__name__:<20} O={o.__name__:<20} kết quả={winner}")

cutoff 2
 Vai  Ván  Thắng   Hòa  Thua       ms/nước
   x  100     96     4     0         1.978
   o  100     92     8     0         1.650
Tổng  200    188    12     0         1.813
cutoff 4
 Vai  Ván  Thắng   Hòa  Thua       ms/nước
   x  100     98     2     0        14.311
   o  100     87    13     0         6.684
Tổng  200    185    15     0        10.597
X=heuristic2_player    O=alpha_beta_player    kết quả=d
X=alpha_beta_player    O=heuristic2_player    kết quả=d
X=heuristic4_player    O=alpha_beta_player    kết quả=d
X=alpha_beta_player    O=heuristic4_player    kết quả=d
X=heuristic2_player    O=heuristic4_player    kết quả=d
X=heuristic4_player    O=heuristic2_player    kết quả=d


Trước Random, cutoff 2 thắng 188/200 ván và cutoff 4 thắng 185/200 ván. Thời gian trung bình mỗi nước tương ứng là 1.813 ms và 10.597 ms. Số ván thắng trước Random đo khả năng tận dụng lỗi ngẫu nhiên, còn kiểm thử đòn hai đường thắng đo chất lượng nước trước phản đòn tối ưu. Kiểm tra chéo từ bàn trống cho 6/6 ván hòa, với mỗi ván đi theo một chuỗi nước xác định.